In [ ]:
import numpy as np
from scipy.special import gammaln # Para calcular log(Gamma(alpha))
from scipy.stats import gamma
import matplotlib.pyplot as plt


class GammaRegression:
    def __init__(self, alpha=0.1, lr=0.001, n_iter=10000, tol=1e-6):
        # Inicializo la clase
        self.alpha = alpha
        self.lr = lr
        self.n_iter = n_iter
        self.tol = tol
        self.theta = None
    def grad(self, X, y, theta):
        sub_eta = X.dot(theta)
        loss = np.sum( np.exp(sub_eta) * y - self.alpha * sub_eta)
        # [:, None] convierte el vector en un vector vertical
        # axis = 0 hace que se sumen los elementos de cada columna
        grad = np.sum((( y * np.exp(sub_eta) - self.alpha )[:, None]) * X, axis=0)
        return loss, grad
    def fit(self, X, y):
        X = np.hstack((np.ones((X.shape[0], 1)), X))
        n,m = X.shape
        # Inicializamos theta en ceros
        theta = np.zeros(m)
        prev_loss = np.inf
        # Hago el desceso por gradiente
        for iteration in range(self.n_iter):
            loss, grad = self.grad(X, y, theta)
            theta = theta - self.lr * grad
            # Verificar convergencia
            if np.abs(prev_loss - loss) < self.tol:
                print(f"Convergencia alcanzada en la iteraci´on {iteration}")
                break
            prev_loss = loss
        self.theta = theta
    def predict(self, X):
        X = np.hstack((np.ones((X.shape[0], 1)), X))
        if self.theta is None:
            raise ValueError("El modelo no ha sido ajustado. Llama a fit() primero.")
        eta = np.exp(X.dot(self.theta))
        return self.alpha/eta
    def loss(self, X, y):
        X = np.hstack((np.ones((X.shape[0], 1)), X))
        eta = np.exp(X.dot(self.theta))
        res = - self.alpha * np.log(eta) - eta*y + (self.alpha - 1)*np.log(y) - gammaln(self.alpha)
        return np.sum(res)


In [ ]:
import pandas as pd
# Importo los datos
Data = pd.read_csv("Datos/waiting_time_data.csv")
X = np.array(Data[["people_in_front","items_in_front"]])
y = np.array(Data["waiting_time"])
model = GammaRegression(alpha=1.1, lr=1e-6, n_iter=10000, tol=1e-8)
model.fit(X, y)
# Grafico los resultados
plt.scatter(y,model.predict(X))
# Calculo el R^2
from sklearn.metrics import r2_score
print(r2_score(y,model.predict(X)))
# 0.98